In [0]:
from pathlib import Path

KNOWLEDGE_DIR = Path(
    "/Workspace/Users/akhileshkumbhar0405@gmail.com/enterprise-analytics-command-center/docs/enterprise_knowledge"
)

print(KNOWLEDGE_DIR)


In [0]:
print("Exists:", KNOWLEDGE_DIR.exists())
print("Is directory:", KNOWLEDGE_DIR.is_dir())

In [0]:
markdown_files = sorted(KNOWLEDGE_DIR.glob("*.md"))

print("Documents found:", len(markdown_files))

for file in markdown_files:
    print("-", file.name)

In [0]:
sample_file = markdown_files[0]

sample_text = sample_file.read_text(encoding="utf-8")

print("File:", sample_file.name)
print("Characters:", len(sample_text))
print()
print(sample_text[:1500])

In [0]:
for file in markdown_files:
    text = file.read_text(encoding="utf-8")

    print(
        f"{file.name:<45}"
        f"{len(text):>8} characters"
    )

In [0]:
expected_files = {
    "01_return_refund_policy.md",
    "02_shipping_fulfillment_policy.md",
    "03_order_cancellation_policy.md",
    "04_product_quality_returns_playbook.md",
    "05_business_metrics_dictionary.md",
    "06_customer_service_operations_guide.md",
}

actual_files = {f.name for f in markdown_files}

missing = expected_files - actual_files
unexpected = actual_files - expected_files

print("Missing:", missing)
print("Unexpected:", unexpected)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.eacc_ecommerce_rag
""")

print("RAG schema ready.")

In [0]:
spark.sql("""
SHOW SCHEMAS IN workspace
""").show(truncate=False)

In [0]:
import re

def extract_document_metadata(text):
    def extract(pattern):
        match = re.search(pattern, text, re.MULTILINE)
        return match.group(1).strip() if match else None

    title = extract(r"^#\s+(.+)$")
    document_owner = extract(r"^Document Owner:\s*(.+)$")
    document_type = extract(r"^Document Type:\s*(.+)$")
    version = extract(r"^Version:\s*(.+)$")
    status = extract(r"^Status:\s*(.+)$")
    last_reviewed = extract(r"^Last Reviewed:\s*(.+)$")

    return {
        "title": title,
        "document_owner": document_owner,
        "document_type": document_type,
        "version": version,
        "status": status,
        "last_reviewed": last_reviewed
    }

In [0]:
metadata_test = extract_document_metadata(sample_text)

metadata_test

In [0]:
documents = []

for file in markdown_files:
    text = file.read_text(encoding="utf-8")
    metadata = extract_document_metadata(text)

    document = {
        "document_id": file.stem,
        "file_name": file.name,
        "title": metadata["title"],
        "document_owner": metadata["document_owner"],
        "document_type": metadata["document_type"],
        "version": metadata["version"],
        "status": metadata["status"],
        "last_reviewed": metadata["last_reviewed"],
        "source_path": str(file),
        "character_count": len(text),
        "content": text
    }

    documents.append(document)

print("Documents prepared:", len(documents))

In [0]:
for doc in documents:
    print(
        doc["document_id"],
        "|",
        doc["title"],
        "|",
        doc["document_owner"],
        "|",
        doc["document_type"],
        "|",
        doc["character_count"]
    )

In [0]:
documents_df = spark.createDataFrame(documents)

display(documents_df)

In [0]:
from pyspark.sql import functions as F

documents_df = (
    documents_df
    .withColumn("ingested_at", F.current_timestamp())
)

In [0]:
display(
    documents_df.select(
        "document_id",
        "title",
        "document_owner",
        "document_type",
        "character_count",
        "ingested_at"
    )
)

In [0]:
(
    documents_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.eacc_ecommerce_rag.raw_documents"
    )
)

print("raw_documents table created.")

In [0]:
spark.sql("""
SELECT
    document_id,
    title,
    document_owner,
    document_type,
    version,
    status,
    character_count,
    ingested_at
FROM workspace.eacc_ecommerce_rag.raw_documents
ORDER BY document_id
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT COUNT(*) AS document_count
FROM workspace.eacc_ecommerce_rag.raw_documents
""").show()

In [0]:
spark.sql("""
SELECT
    document_id,
    LENGTH(content) AS stored_character_count,
    character_count
FROM workspace.eacc_ecommerce_rag.raw_documents
ORDER BY document_id
""").show(truncate=False)

In [0]:
raw_docs_df = spark.table(
    "workspace.eacc_ecommerce_rag.raw_documents"
)

display(
    raw_docs_df.select(
        "document_id",
        "title",
        "document_type",
        "character_count"
    )
)

In [0]:
import re


SECTION_PATTERN = re.compile(
    r"^##\s+(.+?)\s*$",
    re.MULTILINE
)


def parse_markdown_sections(document):
    text = document["content"]

    matches = list(SECTION_PATTERN.finditer(text))

    sections = []

    for index, match in enumerate(matches):
        raw_heading = match.group(1).strip()

        content_start = match.end()

        if index + 1 < len(matches):
            content_end = matches[index + 1].start()
        else:
            content_end = len(text)

        body = text[content_start:content_end].strip()

        # Remove Markdown horizontal-rule lines from section boundaries.
        body = re.sub(
            r"^\s*---\s*$",
            "",
            body,
            flags=re.MULTILINE
        ).strip()

        # Extract numbered section headings such as:
        # "9. Return Rate Monitoring"
        number_match = re.match(
            r"^(\d+)\.\s*(.+)$",
            raw_heading
        )

        if number_match:
            section_number = int(number_match.group(1))
            section_title = number_match.group(2).strip()
        else:
            section_number = None
            section_title = raw_heading

        # Document Metadata is useful for provenance,
        # but we normally don't want it returned as RAG content.
        is_retrievable = (
            raw_heading.lower() != "document metadata"
        )

        section_id = (
            f"{document['document_id']}__"
            f"{index + 1:03d}"
        )

        retrieval_text = (
            f"Document: {document['title']}\n"
            f"Section: {section_title}\n\n"
            f"{body}"
        )

        sections.append({
            "section_id": section_id,
            "document_id": document["document_id"],
            "document_title": document["title"],
            "document_owner": document["document_owner"],
            "document_type": document["document_type"],
            "section_order": index + 1,
            "section_number": section_number,
            "section_heading": raw_heading,
            "section_title": section_title,
            "section_content": body,
            "retrieval_text": retrieval_text,
            "character_count": len(body),
            "is_retrievable": is_retrievable
        })

    return sections

In [0]:
raw_documents = [
    row.asDict()
    for row in raw_docs_df.collect()
]

print("Documents loaded:", len(raw_documents))

In [0]:
test_sections = parse_markdown_sections(
    raw_documents[0]
)

print(
    "Document:",
    raw_documents[0]["title"]
)

print(
    "Sections found:",
    len(test_sections)
)

In [0]:
for section in test_sections:
    print(
        section["section_order"],
        "|",
        section["section_heading"],
        "| retrievable:",
        section["is_retrievable"]
    )

In [0]:
for section in test_sections:
    if section["section_title"] == "Standard Return Window":
        print(section["retrieval_text"])

In [0]:
all_sections = []

for document in raw_documents:
    sections = parse_markdown_sections(document)
    all_sections.extend(sections)

print(
    "Total parsed sections:",
    len(all_sections)
)

In [0]:
from collections import Counter

section_counts = Counter(
    section["document_id"]
    for section in all_sections
)

for document_id, count in sorted(
    section_counts.items()
):
    print(
        document_id,
        "->",
        count
    )

In [0]:
retrievable_sections = [
    section
    for section in all_sections
    if section["is_retrievable"]
]

print(
    "Total sections:",
    len(all_sections)
)

print(
    "Retrievable sections:",
    len(retrievable_sections)
)

In [0]:
sections_df = spark.createDataFrame(
    all_sections
)

display(
    sections_df.select(
        "section_id",
        "document_id",
        "section_order",
        "section_number",
        "section_title",
        "character_count",
        "is_retrievable"
    )
    .orderBy(
        "document_id",
        "section_order"
    )
)

In [0]:
from pyspark.sql import functions as F

sections_df = (
    sections_df
    .withColumn(
        "parsed_at",
        F.current_timestamp()
    )
)

In [0]:
(
    sections_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.eacc_ecommerce_rag.document_sections"
    )
)

print(
    "document_sections table created."
)

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_sections,
    SUM(
        CASE
            WHEN is_retrievable THEN 1
            ELSE 0
        END
    ) AS retrievable_sections,
    SUM(
        CASE
            WHEN NOT is_retrievable THEN 1
            ELSE 0
        END
    ) AS metadata_sections
FROM workspace.eacc_ecommerce_rag.document_sections
""").show()

In [0]:
spark.sql("""
SELECT
    document_title,
    COUNT(*) AS section_count,
    SUM(
        CASE
            WHEN is_retrievable THEN 1
            ELSE 0
        END
    ) AS retrievable_section_count
FROM workspace.eacc_ecommerce_rag.document_sections
GROUP BY document_title
ORDER BY document_title
""").show(truncate=False)

In [0]:
display(
    spark.sql("""
    SELECT
        document_title,
        section_title,
        character_count
    FROM workspace.eacc_ecommerce_rag.document_sections
    WHERE is_retrievable = true
    ORDER BY character_count DESC
    """)
)

In [0]:
retrievable_sections_df = spark.sql("""
SELECT *
FROM workspace.eacc_ecommerce_rag.document_sections
WHERE is_retrievable = true
""")

print(
    "Retrievable sections:",
    retrievable_sections_df.count()
)

In [0]:
from pyspark.sql import functions as F

chunks_df = (
    retrievable_sections_df

    # Each semantic section is currently one chunk.
    .withColumn(
        "chunk_id",
        F.col("section_id")
    )

    .withColumn(
        "chunk_index",
        F.lit(1)
    )

    # This is what we'll eventually embed.
    .withColumn(
        "chunk_text",
        F.col("retrieval_text")
    )

    # Preserve the section body separately from the enriched text.
    .withColumn(
        "source_text",
        F.col("section_content")
    )

    .withColumn(
        "chunk_character_count",
        F.length("retrieval_text")
    )

    # Rough diagnostic only — NOT an exact tokenizer count.
    .withColumn(
        "approx_token_count",
        F.ceil(
            F.length("retrieval_text") / F.lit(4.0)
        ).cast("int")
    )

    # Useful later for detecting changed content.
    .withColumn(
        "content_hash",
        F.sha2(
            F.col("retrieval_text"),
            256
        )
    )

    .withColumn(
        "is_active",
        F.lit(True)
    )

    .withColumn(
        "chunked_at",
        F.current_timestamp()
    )
)

In [0]:
display(
    chunks_df.select(
        "chunk_id",
        "document_title",
        "section_title",
        "chunk_character_count",
        "approx_token_count",
        "content_hash"
    )
    .orderBy(
        "document_id",
        "section_order"
    )
)

In [0]:
display(
    chunks_df
    .filter(
        F.col("section_title")
        == "Standard Return Window"
    )
    .select(
        "chunk_id",
        "document_title",
        "section_title",
        "chunk_text"
    )
)

In [0]:
(
    chunks_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.eacc_ecommerce_rag.chunks"
    )
)

print("RAG chunk table created.")

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT chunk_id) AS unique_chunk_ids
FROM workspace.eacc_ecommerce_rag.chunks
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS empty_chunks
FROM workspace.eacc_ecommerce_rag.chunks
WHERE
    chunk_text IS NULL
    OR TRIM(chunk_text) = ''
""").show()

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS chunk_count,
    MIN(chunk_character_count) AS min_chars,
    ROUND(AVG(chunk_character_count), 2) AS avg_chars,
    MAX(chunk_character_count) AS max_chars,
    MIN(approx_token_count) AS min_approx_tokens,
    ROUND(AVG(approx_token_count), 2) AS avg_approx_tokens,
    MAX(approx_token_count) AS max_approx_tokens
FROM workspace.eacc_ecommerce_rag.chunks
""").show()

In [0]:
spark.sql("""
SELECT
    content_hash,
    COUNT(*) AS duplicate_count
FROM workspace.eacc_ecommerce_rag.chunks
GROUP BY content_hash
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT
    document_title,
    COUNT(*) AS chunk_count
FROM workspace.eacc_ecommerce_rag.chunks
GROUP BY document_title
ORDER BY document_title
""").show(truncate=False)

In [0]:
display(
    spark.sql("""
    SELECT
        chunk_id,
        document_title,
        section_title,
        chunk_character_count,
        approx_token_count
    FROM workspace.eacc_ecommerce_rag.chunks
    ORDER BY chunk_character_count DESC
    LIMIT 15
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        chunk_id,
        document_title,
        section_title
    FROM workspace.eacc_ecommerce_rag.chunks
    WHERE document_owner = 'Customer Operations'
    ORDER BY document_title, section_order
    """)
)

In [0]:
spark.sql("""
SELECT
    SUM(
        CASE
            WHEN chunk_id IS NULL THEN 1 ELSE 0
        END
    ) AS null_chunk_ids,

    SUM(
        CASE
            WHEN document_id IS NULL THEN 1 ELSE 0
        END
    ) AS null_document_ids,

    SUM(
        CASE
            WHEN section_title IS NULL THEN 1 ELSE 0
        END
    ) AS null_section_titles,

    SUM(
        CASE
            WHEN chunk_text IS NULL
                 OR TRIM(chunk_text) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_chunk_texts,

    SUM(
        CASE
            WHEN content_hash IS NULL THEN 1 ELSE 0
        END
    ) AS null_hashes

FROM workspace.eacc_ecommerce_rag.chunks
""").show()